# S09: Reject analysis, FTS vs no-FTS models (spec Chapter 12, features-only branch)

REPORT ONLY. Compares declined applicants (`config.reject_inference.table`) with approved applicants (`config.tables.master`) for Model A and Model C of the FTS model (v1.0.1) and the no-FTS model (1.0.1-nofts), both loaded read-only from their locked registries. Distribution comparison (PSI with each model's own L1 bins, spec 6.4), scoring through the E13 scoring code, extrapolation, swap sets, FTS vs no-FTS agreement and cross-swaps with a paired bootstrap (spec 8.3). No statistical inference (no augmentation / parcelling).

**Writes only** under `<artifacts_root>/<config.reject_inference.output_folder>/` (`ri_reject_fts_compare.xlsx`, `open_questions.md`, `run_summaries/`). Labels are used only for approved applicants with `sample_status` in (GOOD, BAD).

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, pickle, shutil, tempfile, hashlib, importlib, copy, warnings, re
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import lightgbm as lgb
import optbinning
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
RI = cfg["reject_inference"]
U = RI["universe"]

ARTIFACTS = common.artifacts_path(cfg)
OUT = os.path.join(ARTIFACTS, RI["output_folder"])
os.makedirs(os.path.join(OUT, "run_summaries"), exist_ok=True)
XLSX_PATH = os.path.join(OUT, "ri_reject_fts_compare.xlsx")
OQ_PATH = os.path.join(OUT, "open_questions.md")
SUMMARY_PATH = os.path.join(OUT, "run_summaries", "s09_ri_reject_fts_compare.json")

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.{cfg['tables']['master']}"
RI_TBL, OVERLAP_TBL = RI["table"], RI["overlap_check_table"]
# E13 stored case-level scores (s05_oot 'Score full OOT window'), used only for the 1,000-row spot check
E13_STORED = {"FTS": f"{cat}.{sch}.acard_oot_predictions", "NOFTS": f"{cat}.{sch}.acard_oot_predictions_v101_nofts"}
id_col = cfg["columns"]["id"]
DATE_A, DECISION_A = cfg["columns"]["date"], cfg["columns"]["decision_time"]
LABELLED = cfg["labelled_statuses"]
EPS = cfg["funnel"]["L3"]["eps"]
PSI_GREEN, PSI_RED = cfg["funnel"]["L3"]["psi_green"], cfg["funnel"]["L3"]["psi_red"]
WINDOWS = list(cfg["windows"])
BOOT = cfg["selection"]["bootstrap"]
PROPOSED_CUTOFF = cfg.get("strategy", {}).get("proposed_cutoff")
WATCH = cfg["column_roles"]["watch"]
SEGMENTS = cfg["columns"]["segments"] or []
SWITCH = pd.Timestamp(U["switch_date"])
SWITCH_MONTH = SWITCH.strftime("%Y-%m")
RULES = ["A", "B", "B_FALLBACK"]
VARIANT_MODELS = ["A", "C"]            # owner 2026-11-16: compare Model A and Model C in both scenarios
DEC_VALUES = [str(v).strip().upper() for v in U["decision_values"]]
SEG_VALUES = [str(v).strip().upper() for v in U["risk_segment_values"]]
REASON = str(U["reason_contains"]).strip().upper()

# Report-only constants from the request (nothing is selected, dropped or fitted with them)
SWITCH_DIAG_FROM = "2024-01"           # STEP 0(d) / 1.2 monthly view
TRUNC_APPROVED_MAX, TRUNC_DECLINED_MIN = 0.05, 0.20     # STEP 2.3
EXTRAP_Q, EXTRAP_SPARSE_BIN = (0.01, 0.99), 0.01         # STEP 3.3
DISAGREE_DECILES = 3                   # STEP 4.5
AGREEMENT_FLAG_GAP = 0.05              # STEP 4.1 "clearly lower" (open question)
AGE_BANDS = [0, 25, 30, 35, 40, 45, 50, 60, 200]
TOP_N_WATCH = 10
HARD_POLICY = {"BLACKLIST": r"BLACK ?LIST", "FRAUD": r"FRAUD", "KYC": r"\bKYC\b", "AGE": r"\bAGE\b|UNDER ?AGE",
               "DOCUMENT": r"DOCUMENT|\bDOCS?\b", "EMPLOYER": r"EMPLOYER|EMPLOYMENT|COMPANY",
               "WITHDRAWN": r"WITHDRAW", "DUPLICATE": r"DUPLICAT"}

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
RUN_TIME = datetime.now().isoformat(timespec="seconds")
_norm = common._norm                   # E2b normalisation: text -> trim -> UPPER, "" = null
WRITTEN = []

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return str(o)

def _guard(dst):
    assert os.path.abspath(dst).startswith(os.path.abspath(OUT) + os.sep), f"refusing to write outside {OUT}: {dst}"

def write_text(text, dst):
    """Local temp file, then copy to the UC volume (copyfile: FUSE does not allow chmod/utime)."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s09_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        fh.write(text)
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def write_json(obj, dst):
    write_text(json.dumps(obj, indent=2, default=_json_default), dst)

def save_excel(sheets, dst):
    """sheets: name -> DataFrame, or list of (title, DataFrame) blocks stacked vertically on one sheet."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s09_"), os.path.basename(dst))
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, obj in sheets.items():
            assert len(name) <= 31, f"sheet name too long: {name}"
            blocks = obj if isinstance(obj, list) else [(None, obj)]
            r = 0
            for title, df in blocks:
                df = df.copy()
                for c in df.columns:
                    if isinstance(df[c].dtype, pd.PeriodDtype) or str(df[c].dtype) in ("string", "category"):
                        df[c] = df[c].astype(str)
                if title:
                    pd.DataFrame(columns=[title]).to_excel(xw, sheet_name=name, startrow=r, index=False)
                    r += 1
                df.to_excel(xw, sheet_name=name, startrow=r, index=False)
                r += len(df) + 3
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def psi_status(x):
    return "GREEN" if x < PSI_GREEN else ("RED" if x >= PSI_RED else "AMBER")

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}\nOutput      : {OUT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | switch {SWITCH.date()} | reason '{REASON}'")

In [0]:
FDICT_PATH = os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")
FDICT = pd.read_csv(FDICT_PATH)
FDICT_SHA = sha256_file(FDICT_PATH)

MODELS = {}
for mc in RI["models"]:
    name, reg_path = mc["name"], mc["registry"]
    assert os.path.exists(reg_path), f"ABORT {name}: registry not found: {reg_path}"
    reg = read_json(reg_path)
    assert reg.get("locked") is True, f"ABORT {name}: registry is not locked"
    for k in ("decision_rules_signed_by", "signed_date"):
        assert str(reg.get("signoff", {}).get(k, "")).strip(), f"ABORT {name}: sign-off field {k} is empty"
    bad = [k for k, v in reg["files"].items() if not os.path.exists(v["path"]) or sha256_file(v["path"]) != v["sha256"]]
    assert not bad, f"ABORT {name}: {len(bad)} locked file(s) changed since the lock: {bad}"
    root = os.path.dirname(reg_path)
    files = {k: v["path"] for k, v in reg["files"].items()}
    # Only the frozen folds may be shared from another folder (no-FTS reuses the FTS folds); hashes are verified above
    outside = [k for k, p in files.items() if not p.startswith(root + "/")]
    assert all(k.startswith("folds/") for k in outside), f"ABORT {name}: registry points outside its own folder: {outside}"
    platt_path = os.path.join(root, "calibration", "platt_dev_oof.json")
    platt_doc = read_json(platt_path)
    assert platt_doc["registry_version"] == reg["version"], f"ABORT {name}: Platt file belongs to another registry version"
    spec = read_json(files["models/model_spec.json"])
    A_FEATS, POOL = reg["features"]["A"], reg["features"]["BC_pool"]
    FEATS = list(dict.fromkeys(A_FEATS + POOL))
    SEL = common.resolve_selected_model(reg, verbose=False)   # model_choice_override if present, else E13 rule result
    print(f"{name}: selected model {SEL['model']} (source: {SEL['source']}; E13 rule selection {SEL['rule_selected_model']})")
    C_BST = lgb.Booster(model_file=files["models/C/lgbm.txt"])
    assert C_BST.num_trees() == reg["lightgbm_rounds"]["final_rounds"], f"ABORT {name}: LightGBM trees != locked final rounds"
    MODELS[name] = {
        "REG_PATH": reg_path, "REG_SHA": sha256_file(reg_path), "ROOT": root, "FILES": files,
        "VERSION": reg["version"], "LOCKED_AT": reg["locked_at"],
        "SELECTED": SEL["model"], "STATUS": SEL["status"], "SELECTION": SEL,
        "PLATT_PATH": platt_path, "PLATT_SHA": sha256_file(platt_path), "PLATT": platt_doc["params"],
        "WOE_KW": spec["A"]["woe_transform"], "A_FEATS": A_FEATS, "POOL": POOL, "FEATS": FEATS,
        "BC_COLS": reg["features"]["BC_columns"],
        "A_BP": pickle.load(open(files["models/A/binning_process.pkl"], "rb")),
        "A_LR": pickle.load(open(files["models/A/lr_sklearn.pkl"], "rb")),
        "C_BST": C_BST,
        "L1": {f: pickle.load(open(files[f"binning/L1/{f}.pkl"], "rb")) for f in FEATS},
        "FROZEN": set(pd.read_parquet(files["folds/r_folds.parquet"], columns=["app_id"])["app_id"].astype(str)),
        "N_FILES": len(files),
    }
    print(f"{name}: v{reg['version']} locked {reg['locked_at']} | selected {MODELS[name]['SELECTED']} ({MODELS[name]['STATUS']}) | "
          f"A {len(A_FEATS)} feats, C pool {len(POOL)} | {len(files)} hashes OK | Platt A {platt_doc['params']['A']['a']:.4f}/{platt_doc['params']['A']['b']:.4f}")

for name, M in MODELS.items():
    if name != "FTS":
        assert not os.path.abspath(OUT).startswith(os.path.abspath(M["ROOT"]) + os.sep), f"output folder is inside the {name} model folder"

VARIANTS = [f"{s}_{m}" for s in MODELS for m in VARIANT_MODELS]
VFEATS = {f"{s}_A": M["A_FEATS"] for s, M in MODELS.items()} | {f"{s}_C": M["POOL"] for s, M in MODELS.items()}
scen = lambda v: v.rsplit("_", 1)[0]

def tree_snapshot(roots):
    """(size, mtime) of every file under the model folders, excluding the output folder."""
    snap, out_abs = {}, os.path.abspath(OUT)
    for r in roots:
        for dp, _, fn in os.walk(r):
            if os.path.abspath(dp) == out_abs or os.path.abspath(dp).startswith(out_abs + os.sep):
                continue
            for f in fn:
                p = os.path.join(dp, f)
                st = os.stat(p)
                snap[p] = (st.st_size, st.st_mtime)
    return snap

def table_version(t):
    return int(spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"])

PROTECTED_TABLES = {}
for t in [MASTER_TBL, RI_TBL, OVERLAP_TBL, f"{cat}.{sch}.acard_feature_log", f"{cat}.{sch}.{cfg['tables']['feature_log']}"]:
    if spark.catalog.tableExists(t):
        PROTECTED_TABLES[t] = table_version(t)
SNAP0 = tree_snapshot([M["ROOT"] for M in MODELS.values()])
print(f"Snapshot: {len(SNAP0)} files under the model folders | protected tables {PROTECTED_TABLES}")

In [0]:
FEATS_ALL = list(dict.fromkeys(f for M in MODELS.values() for f in M["FEATS"]))
FTS_FLAG = RI["fts_flag_column"] if isinstance(RI["fts_flag_column"], list) else [RI["fts_flag_column"]]

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c in cols:
            return c
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

def watch_source(cols, w):
    return pick(cols, [w] + (["gender"] if w.upper() in ("SEX", "GENDER") else []))

m_all_cols = spark.table(MASTER_TBL).columns
MFLAG = pick(m_all_cols, FTS_FLAG)
assert MFLAG, f"no fts_flag_column {FTS_FLAG} in {MASTER_TBL}"
M_WATCH = {w: watch_source(m_all_cols, w) for w in WATCH}
m_keep = list(dict.fromkeys([id_col, "CREDITCARDNO", "EMID_REFERENCENUMBER", DATE_A, DECISION_A, "window", "sample_status", MFLAG]
                            + [c for c in M_WATCH.values() if c] + [c for c in SEGMENTS if c in m_all_cols] + FEATS_ALL))
master = spark.table(MASTER_TBL).select(*m_keep).toPandas()

RI_SDF = spark.table(RI_TBL)
ri_all_cols = RI_SDF.columns
RFLAG = pick(ri_all_cols, FTS_FLAG)
assert RFLAG, f"no fts_flag_column {FTS_FLAG} in {RI_TBL}"
R_WATCH = {w: watch_source(ri_all_cols, w) for w in WATCH}
missing_feats = [f for f in FEATS_ALL if f not in ri_all_cols]
assert not missing_feats, f"model features missing in {RI_TBL}: {missing_feats}"
ri_raw = RI_SDF.toPandas()

# Approved applicants: master, sample_status not EXCLUDED, inside a window
master["has_fts_data"] = np.where(_norm(master[MFLAG]).notna(), "Y", "N")
_d = pd.to_datetime(master[DATE_A])
master["app_month"] = _d.dt.to_period("M").astype(str)
master["app_quarter"] = _d.dt.to_period("Q").astype(str)
P = master[(master["sample_status"] != "EXCLUDED") & master["window"].isin(WINDOWS)].reset_index(drop=True)
DEV_GB = master[(master["window"] == "DEV") & master["sample_status"].isin(LABELLED)].reset_index(drop=True)
print(f"master {master.shape} | approved in scope {len(P):,} by window {P['window'].value_counts().to_dict()}")
print(f"RI table {ri_raw.shape} | fts flag: master {MFLAG}, RI {RFLAG} | watch master {M_WATCH} RI {R_WATCH}")

In [0]:
ri = ri_raw.copy()
# Key: AppKey (unique; 367 application_id values are shared by different applications, see open questions)
_k = ri["AppKey"].astype("string")
ri["app_id"] = _k.where(_k.notna(), "APPID_" + ri["application_id"].astype("string"))
assert ri["app_id"].is_unique, "app_id (AppKey) is not unique"

ri["_date"] = pd.to_datetime(ri[U["period_column"]])
ri["_dec"] = _norm(ri[U["decision_column"]])
ri["_seg"] = _norm(ri[U["risk_segment_column"]])
ri["_reason"] = _norm(ri[U["reason_column"]])
ri["app_month"] = ri["_date"].dt.to_period("M").astype(str)

dec_ok = ri["_dec"].isin(DEC_VALUES).to_numpy() if U["require_decision"] else np.ones(len(ri), bool)
has_reason = ri["_reason"].str.contains(REASON, regex=False).fillna(False).astype(bool).to_numpy()
pre = (ri["_date"] < SWITCH).to_numpy()
post = (ri["_date"] >= SWITCH).to_numpy()
seg_ne = ri["_seg"].notna().to_numpy()
seg_in = ri["_seg"].isin(SEG_VALUES).to_numpy()
COND = {"A": pre & has_reason, "B": post & seg_ne & seg_in, "B_FALLBACK": post & ~seg_ne & has_reason}
ri["ri_rule"] = None
for r in RULES:
    ri.loc[dec_ok & COND[r], "ri_rule"] = r

UNIVERSE_TOTAL = pd.DataFrame(
    [{"step": "all rows in input table", "n": len(ri)},
     {"step": f"{U['decision_column']} in {DEC_VALUES}", "n": int(dec_ok.sum())}]
    + [{"step": f"rule {r}", "n": int((ri['ri_rule'] == r).sum())} for r in RULES]
    + [{"step": "RI universe (any rule)", "n": int(ri["ri_rule"].notna().sum())}])

monthly = ri.groupby("app_month").agg(n_rows=("app_id", "size"))
monthly["n_decision_ok"] = pd.Series(dec_ok, index=ri.index).groupby(ri["app_month"]).sum()
for r in RULES:
    monthly[r] = ri[ri["ri_rule"] == r].groupby("app_month").size()
monthly = monthly.fillna(0).astype(int)
monthly["universe"] = monthly[RULES].sum(axis=1)
monthly = monthly.reset_index()
monthly["period"] = np.where(monthly["app_month"] >= SWITCH_MONTH, "from_switch", "before_switch")
monthly["switch_marker"] = np.where(monthly["app_month"] == SWITCH_MONTH, "SWITCH DATE", "")

# Diagnostics (report only): from switch_date, segment other than the rule-B values that contain the rule-A reason
diag = post & seg_ne & ~seg_in & has_reason
SWITCH_DIAG = (ri[diag].assign(decision_ok=dec_ok[diag]).groupby(["_seg", "decision_ok"]).size()
               .reset_index(name="n").rename(columns={"_seg": "risk_segment"}))
if SWITCH_DIAG.empty:
    SWITCH_DIAG = pd.DataFrame([{"risk_segment": "(none)", "decision_ok": None, "n": 0}])
SEG_X_DEC = (ri[post].assign(segment=ri.loc[post, "_seg"].fillna("(EMPTY)"), decision=ri.loc[post, "_dec"].fillna("(EMPTY)"))
             .groupby(["segment", "decision"]).size().unstack(fill_value=0).reset_index())
display(UNIVERSE_TOTAL)
display(monthly[monthly["app_month"] >= SWITCH_DIAG_FROM])
display(SWITCH_DIAG)

In [0]:
def resolve_col(c, cols):
    if c in cols:
        return c, "exact"
    low = {x.lower(): x for x in cols}
    if c.lower() in low:
        return low[c.lower()], "case-insensitive"
    if c.lower().startswith("aecb_") and c[5:].lower() in low:
        return low[c[5:].lower()], "alias: aecb_ prefix dropped in RI build"
    return None, "NOT_AVAILABLE in RI table"

# ── 1.3 application-time population filters, waterfall by rule ──
uni = ri[ri["ri_rule"].notna()].copy()
APPLY = RI.get("population_filters_mode", "apply") == "apply"
alive = pd.Series(True, index=uni.index)
uni["ri_exclusion_reason"] = None
rule_of = uni["ri_rule"].to_numpy()

def wf_row(order, step, cfg_col, col, how, fail):
    row = {"order": order, "step": step, "column_in_config": cfg_col, "column_used": col, "resolution": how}
    for r in RULES:
        mr = rule_of == r
        row[f"removed_{r}"] = int((fail.to_numpy() & mr).sum())
        row[f"remaining_{r}"] = int((alive.to_numpy() & mr).sum())
    row["removed_total"], row["remaining_total"] = int(fail.sum()), int(alive.sum())
    return row

nofail = pd.Series(False, index=uni.index)
WF = [wf_row(0, "RI universe (rules A / B / B_FALLBACK)", None, None, "", nofail)]
order = 0
for kind in ("exclude", "include"):
    for rule in cfg["population_filters"]["application_time"].get(kind, []):
        order += 1
        col, how = resolve_col(rule["column"], list(uni.columns))
        if col is None or not APPLY:
            fail = nofail.copy()
            how = how if APPLY else how + " (mode != apply)"
        else:
            hit = pd.Series(common._rule_mask(pd.DataFrame({rule["column"]: uni[col]}), rule).to_numpy(), index=uni.index)
            fail = alive & (hit if kind == "exclude" else ~hit)
        uni.loc[fail, "ri_exclusion_reason"] = rule["name"]
        alive &= ~fail
        WF.append(wf_row(order, f"application_time:{kind}:{rule['name']}", rule["column"], col, how, fail))
for kind in ("exclude", "include"):
    for rule in cfg["population_filters"]["performance_related"].get(kind, []):
        order += 1
        WF.append(wf_row(order, f"performance_related:{kind}:{rule['name']}", rule["column"], None,
                         "not applied: performance-related, does not apply to declines", nofail))
WATERFALL = pd.DataFrame(WF)
uni_in = uni[alive.to_numpy()].copy()

# ── 1.4 decline reasons (report only) ──
rs = uni_in[["app_id", "ri_rule"]].assign(reason=uni_in["_reason"].astype(object).str.split(RI["reason_delimiter"], regex=False)).explode("reason")
rs["reason"] = rs["reason"].astype("string").str.strip()
rs = rs[rs["reason"].notna() & (rs["reason"] != "")]
REASONS = rs.groupby(["reason", "ri_rule"])["app_id"].nunique().unstack(fill_value=0).reindex(columns=RULES, fill_value=0)
REASONS["total"] = REASONS.sum(axis=1)
REASONS = REASONS.sort_values("total", ascending=False).reset_index()
REASONS["hard_policy_flag"] = REASONS["reason"].map(
    lambda t: ",".join(k for k, pat in HARD_POLICY.items() if re.search(pat, str(t))) or "")
REASONS["share_of_universe"] = REASONS["total"] / len(uni_in)

# ── 1.5 windows from application_date ──
cfg_ri = copy.deepcopy(cfg)
cfg_ri["columns"]["date"] = U["period_column"]
uni_in = common.assign_window(uni_in, cfg_ri)
uni_in["window"] = uni_in["window"].astype(object).where(uni_in["window"].notna(), "OUT_OF_SCOPE")
uni_in["app_month"] = uni_in["app_month"].astype(str)
uni_in["app_quarter"] = uni_in["app_quarter"].astype(str)

# ── 1.7 has_fts_data (reference number, segment only) ──
uni_in["has_fts_data"] = np.where(_norm(uni_in[RFLAG]).notna(), "Y", "N")

# ── 1.8 declined first, approved later (customer link: emirates_id = EMID_REFERENCENUMBER) ──
appr_link = master.assign(_emid=_norm(master["EMID_REFERENCENUMBER"]))[["_emid", "CREDITCARDNO", DATE_A, DECISION_A, "sample_status", "window"]]
appr_link = appr_link[appr_link["_emid"].notna()]
lk = uni_in[["app_id", "ri_rule", "window", "_date"]].assign(_emid=_norm(uni_in["emirates_id"])).dropna(subset=["_emid"])
lk = lk.merge(appr_link, on="_emid", suffixes=("", "_appr"))
lk = lk[pd.to_datetime(lk[DATE_A]) > lk["_date"]].sort_values(DATE_A).drop_duplicates("app_id")
lk["days_to_later_application"] = (pd.to_datetime(lk[DATE_A]) - lk["_date"]).dt.days
uni_in["reapplied_later_approved"] = np.where(uni_in["app_id"].isin(lk["app_id"]), "Y", "N")

def reapply_rows(df, level_type, level):
    gb = df[df["sample_status"].isin(LABELLED)]
    return {"level_type": level_type, "level": level, "declined_later_approved": len(df),
            "distinct_customers": df["_emid"].nunique() if len(df) else 0,
            "later_good_bad": len(gb), "later_bads": int((gb["sample_status"] == "BAD").sum()),
            "later_bad_rate": (gb["sample_status"] == "BAD").mean() if len(gb) else np.nan,
            "median_days_to_later_application": df["days_to_later_application"].median() if len(df) else np.nan,
            **{f"later_status_{s}": int((df["sample_status"] == s).sum()) for s in sorted(master["sample_status"].dropna().unique())}}
REAPPLY = [reapply_rows(lk, "ALL", "ALL")]
for kt in ("ri_rule", "window"):
    for kv, g in lk.groupby(kt):
        REAPPLY.append(reapply_rows(g, kt, kv))
for kv, g in lk.groupby("window_appr"):
    REAPPLY.append(reapply_rows(g, "window_of_later_approval", kv))
REAPPLY = pd.DataFrame(REAPPLY)
_gb_all = P[P["sample_status"].isin(LABELLED)]
REAPPLY_BENCH = pd.DataFrame([{"population": "all approved GOOD+BAD (master, in scope)", "n": len(_gb_all),
                               "bads": int((_gb_all["sample_status"] == "BAD").sum()),
                               "bad_rate": (_gb_all["sample_status"] == "BAD").mean()}])

# In-scope declined applicants (window not OUT_OF_SCOPE)
D = uni_in[uni_in["window"] != "OUT_OF_SCOPE"].reset_index(drop=True)

# ── 1.6 timing gap for approved applicants ──
_gap = (pd.to_datetime(P[DECISION_A]) - pd.to_datetime(P[DATE_A])).dt.days
TIMING = (pd.DataFrame({"window": P["window"], "gap_days": _gap}).groupby("window")["gap_days"]
          .describe(percentiles=[0.05, 0.5, 0.95]).reset_index())
TIMING["share_negative"] = TIMING["window"].map(pd.Series(_gap < 0).groupby(P["window"]).mean())

print(f"universe {len(uni):,} -> after application-time filters {len(uni_in):,} -> in scope {len(D):,}")
print("OUT_OF_SCOPE:", int((uni_in['window'] == 'OUT_OF_SCOPE').sum()), "| by window:", D["window"].value_counts().to_dict())
display(WATERFALL)
display(REAPPLY.head(12)); display(REAPPLY_BENCH)

In [0]:
ov_ids = set(spark.table(OVERLAP_TBL).select("application_id").toPandas()["application_id"].dropna().astype(str))
ri_ids = set(ri["application_id"].dropna().astype(str))
vc = ri["application_id"].astype(str).value_counts()
TABLE_OVERLAP = pd.DataFrame([
    {"item": "distinct application_id in RI table (acard_master_ri_fts)", "n": len(ri_ids)},
    {"item": "distinct application_id in overlap_check_table (acard_master_ri)", "n": len(ov_ids)},
    {"item": "in both", "n": len(ri_ids & ov_ids)},
    {"item": "only in RI table", "n": len(ri_ids - ov_ids)},
    {"item": "only in overlap_check_table", "n": len(ov_ids - ri_ids)},
    {"item": "RI universe (after filters) application_id in overlap_check_table", "n": int(uni_in["application_id"].astype(str).isin(ov_ids).sum())},
    {"item": "RI universe (after filters) application_id NOT in overlap_check_table", "n": int((~uni_in["application_id"].astype(str).isin(ov_ids)).sum())},
    {"item": "application_id values shared by >1 row (kept: different AppKey / dates)", "n": int((vc > 1).sum())},
    {"item": "rows with a shared application_id", "n": int(vc[vc > 1].sum())},
    {"item": "AppKey null (key falls back to application_id)", "n": int(ri["AppKey"].isna().sum())},
])

FTS_MODEL_FEATS = [f for f in FEATS_ALL if f.lower().startswith("fts_")]
def cov_row(df, group, kt, kv):
    return {"group": group, "level_type": kt, "level": kv, "n": len(df),
            "share_has_fts_data_Y": (df["has_fts_data"] == "Y").mean() if len(df) else np.nan,
            "share_any_fts_model_feature": df[FTS_MODEL_FEATS].notna().any(axis=1).mean() if len(df) else np.nan}
FTS_COV = [cov_row(P, "approved", "ALL", "ALL"), cov_row(D, "declined", "ALL", "ALL")]
for w in WINDOWS:
    FTS_COV += [cov_row(P[P["window"] == w], "approved", "window", w), cov_row(D[D["window"] == w], "declined", "window", w)]
for r in RULES:
    FTS_COV.append(cov_row(D[D["ri_rule"] == r], "declined", "ri_rule", r))
FTS_COV = pd.DataFrame(FTS_COV)
display(TABLE_OVERLAP); display(FTS_COV)

In [0]:
def score_variants(df):
    """E13 scoring (s05_oot 'Score with the locked models'): raw PD -> Platt (DEV OOF) -> score, per model folder."""
    out = {}
    for s, M in MODELS.items():
        XA = df[M["A_FEATS"]].apply(pd.to_numeric, errors="coerce")
        pa = M["A_LR"].predict_proba(np.asarray(M["A_BP"].transform(XA, metric="woe", **M["WOE_KW"]), dtype=float))[:, 1]
        XR = common.prepare_raw(df, M["POOL"], FDICT)
        assert list(XR.columns) == M["BC_COLS"], f"{s}: prepare_raw columns differ from the locked B/C columns"
        pc = M["C_BST"].predict(XR)
        for m, p in (("A", pa), ("C", pc)):
            cal = common.apply_platt(p, M["PLATT"][m])
            out[f"{s}_{m}"] = pd.DataFrame({"raw": p, "pd": cal, "score": common.pd_to_score(cal, cfg)})
    return out

SC_P, SC_D, SC_DEV = score_variants(P), score_variants(D), score_variants(DEV_GB)
for nm, SC in (("approved", SC_P), ("declined", SC_D)):
    for v in VARIANTS:
        x = SC[v]["pd"].to_numpy()
        assert np.isfinite(x).all() and ((x > 0) & (x < 1)).all(), f"{nm} {v}: PD outside (0, 1)"

# DEV cut-points: deciles of each model's calibrated PD on its own frozen DEV GOOD+BAD (as E13)
def band(p, e):
    return np.searchsorted(e, p, side="right") + 1          # 1 = lowest PD (best) ... 10 = highest PD
EDGES, N_DEV = {}, {}
for v in VARIANTS:
    mk = DEV_GB[id_col].astype(str).isin(MODELS[scen(v)]["FROZEN"]).to_numpy()
    N_DEV[v] = int(mk.sum())
    EDGES[v] = np.unique(np.quantile(SC_DEV[v]["pd"].to_numpy()[mk], np.linspace(0, 1, 11)[1:-1]))
DEC_P = {v: band(SC_P[v]["pd"].to_numpy(), EDGES[v]) for v in VARIANTS}
DEC_D = {v: band(SC_D[v]["pd"].to_numpy(), EDGES[v]) for v in VARIANTS}

# ── Spot check: approved OOT scores reproduce the stored E13 scores (1,000 rows) ──
stored = {}
for s, tbl in E13_STORED.items():
    st = spark.table(tbl).select("CREDITCARDNO", *[f"prob_{m}_cal" for m in VARIANT_MODELS], *[f"score_{m}" for m in VARIANT_MODELS]).toPandas()
    st["CREDITCARDNO"] = st["CREDITCARDNO"].astype(str)
    stored[s] = st[~st["CREDITCARDNO"].duplicated(keep=False)].set_index("CREDITCARDNO")
oot_card = P.loc[P["window"] == "OOT", "CREDITCARDNO"].astype(str)
oot_card = oot_card[~oot_card.duplicated(keep=False)]
candidates = sorted(set(oot_card) & set.intersection(*[set(st.index) for st in stored.values()]))
spot_ids = set(np.random.default_rng(SEED).choice(candidates, size=min(1000, len(candidates)), replace=False))
sel = oot_card.index[oot_card.isin(spot_ids)].to_numpy()
ids_sel = P.loc[sel, "CREDITCARDNO"].astype(str)
SPOT = []
for v in VARIANTS:
    s, m = scen(v), v.rsplit("_", 1)[1]
    st = stored[s].loc[ids_sel]
    d_pd = np.abs(SC_P[v]["pd"].to_numpy()[sel] - st[f"prob_{m}_cal"].to_numpy())
    SPOT.append({"variant": v, "stored_table": E13_STORED[s], "n": len(sel), "max_abs_pd_diff": float(d_pd.max()),
                 "n_score_diff": int((SC_P[v]["score"].to_numpy()[sel] != st[f"score_{m}"].to_numpy()).sum())})
SPOT = pd.DataFrame(SPOT)
print("DEV GOOD+BAD (frozen) per variant:", N_DEV)
display(SPOT)

In [0]:
def bins_of(optb, x):
    if str(getattr(optb, "dtype", "numerical")) == "numerical":
        arr = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)
    else:
        arr = x.astype(object).where(x.notna(), np.nan).to_numpy()
    return pd.Series(np.asarray(optb.transform(arr, metric="bins")).astype(str))

BINS = {s: {f: (bins_of(M["L1"][f], P[f]), bins_of(M["L1"][f], D[f])) for f in M["FEATS"]} for s, M in MODELS.items()}
wP = {w: (P["window"] == w).to_numpy() for w in WINDOWS}
wD = {w: (D["window"] == w).to_numpy() for w in WINDOWS}
rD = {"ALL": np.ones(len(D), bool)} | {r: (D["ri_rule"] == r).to_numpy() for r in RULES}
QUARTERS = sorted(set(P["app_quarter"]) | set(D["app_quarter"]))
qP = {q: (P["app_quarter"] == q).to_numpy() for q in QUARTERS}
qD = {q: (D["app_quarter"] == q).to_numpy() for q in QUARTERS}

PSI_WIN, PSI_Q = {}, {}
for s, M in MODELS.items():
    rows, qrows = [], []
    for f in M["FEATS"]:
        bp, bd = BINS[s][f]
        for w in WINDOWS:
            if wP[w].sum() == 0:
                continue
            for lvl, mr in rD.items():
                md = wD[w] & mr
                if md.sum():
                    rows.append({"feature": f, "window": w, "ri_rule": lvl, "comparison": "declined_vs_approved",
                                 "n_approved": int(wP[w].sum()), "n_declined": int(md.sum()),
                                 "psi": common.psi_from_bins(bp[wP[w]], bd[md], EPS)})
            if wD[w].sum():
                rows.append({"feature": f, "window": w, "ri_rule": "ALL", "comparison": "ttd_vs_approved (diagnosis)",
                             "n_approved": int(wP[w].sum()), "n_declined": int(wD[w].sum()),
                             "psi": common.psi_from_bins(bp[wP[w]], pd.concat([bp[wP[w]], bd[wD[w]]], ignore_index=True), EPS)})
        for q in QUARTERS:
            if qP[q].sum() == 0:
                continue
            for lvl, mr in rD.items():
                md = qD[q] & mr
                if md.sum():
                    qrows.append({"feature": f, "app_quarter": q, "ri_rule": lvl, "n_approved": int(qP[q].sum()),
                                  "n_declined": int(md.sum()), "psi": common.psi_from_bins(bp[qP[q]], bd[md], EPS),
                                  "from_switch": q >= str(SWITCH.to_period("Q"))})
    PSI_WIN[s] = pd.DataFrame(rows).assign(status=lambda d: d["psi"].map(psi_status))
    PSI_Q[s] = pd.DataFrame(qrows).assign(status=lambda d: d["psi"].map(psi_status))

# Missing rates by group and quarter / window / rule; FTS features also by has_fts_data
def miss_long(df, group, dim, feats):
    key = df[dim].astype(str) if dim != "_all" else pd.Series("ALL", index=df.index)
    g = df[feats].isna().groupby(key).mean()
    n = key.value_counts()
    out = g.reset_index(names="value").melt(id_vars="value", var_name="feature", value_name="missing_rate")
    out["n"] = out["value"].map(n)
    out.insert(0, "dim", "ALL" if dim == "_all" else dim)
    out.insert(0, "group", group)
    return out
MISSING = pd.concat([miss_long(P, "approved", d, FEATS_ALL) for d in ("_all", "window", "app_quarter")]
                    + [miss_long(D, "declined", d, FEATS_ALL) for d in ("_all", "window", "app_quarter", "ri_rule")]
                    + [miss_long(P, "approved", "has_fts_data", FTS_MODEL_FEATS), miss_long(D, "declined", "has_fts_data", FTS_MODEL_FEATS)],
                    ignore_index=True)
MISSING["used_by"] = MISSING["feature"].map(lambda f: ",".join(v for v in VARIANTS if f in VFEATS[v]))

# Policy-truncated bins: < 5% of approved and > 20% of declined
TRUNC = []
for s, M in MODELS.items():
    for f in M["FEATS"]:
        bp, bd = BINS[s][f]
        for w, mp, md in [("ALL", np.ones(len(P), bool), np.ones(len(D), bool))] + [(w, wP[w], wD[w]) for w in WINDOWS]:
            if mp.sum() == 0 or md.sum() == 0:
                continue
            sp, sd = bp[mp].value_counts(normalize=True), bd[md].value_counts(normalize=True)
            for b in sp.index.union(sd.index):
                a_, d_ = float(sp.get(b, 0.0)), float(sd.get(b, 0.0))
                TRUNC.append({"scenario": s, "feature": f, "window": w, "bin": b, "share_approved": a_, "share_declined": d_,
                              "n_approved": int(mp.sum()), "n_declined": int(md.sum()),
                              "policy_truncated": (a_ < TRUNC_APPROVED_MAX) and (d_ > TRUNC_DECLINED_MIN)})
TRUNC = pd.DataFrame(TRUNC)
TRUNC_FEATS = TRUNC[TRUNC["policy_truncated"] & (TRUNC["window"] == "ALL")].groupby("scenario")["feature"].unique()

# Through-the-door monthly series (approved + RI-universe declines), switch date marked
months = sorted(set(P["app_month"]) | set(D["app_month"]))
TTD = pd.DataFrame({"app_month": months})
TTD["approved"] = TTD["app_month"].map(P["app_month"].value_counts()).fillna(0).astype(int)
TTD["declined_ri"] = TTD["app_month"].map(D["app_month"].value_counts()).fillna(0).astype(int)
for r in RULES:
    TTD[f"declined_{r}"] = TTD["app_month"].map(D.loc[D["ri_rule"] == r, "app_month"].value_counts()).fillna(0).astype(int)
TTD["approval_rate"] = TTD["approved"] / (TTD["approved"] + TTD["declined_ri"]).replace(0, np.nan)
TTD["period"] = np.where(TTD["app_month"] >= SWITCH_MONTH, "from_switch", "before_switch")
TTD["switch_marker"] = np.where(TTD["app_month"] == SWITCH_MONTH, "SWITCH DATE", "")

for s in MODELS:
    t = PSI_WIN[s][(PSI_WIN[s]["ri_rule"] == "ALL") & (PSI_WIN[s]["comparison"] == "declined_vs_approved")]
    print(s, "declined vs approved PSI status counts (ALL rules):", t["status"].value_counts().to_dict())
print("policy-truncated features (overall):", {k: list(v) for k, v in TRUNC_FEATS.items()})

In [0]:
def level_masks(df, kinds):
    out = [("ALL", "ALL", np.ones(len(df), bool))]
    for k in kinds:
        vals = df[k].astype(str)
        for val in sorted(vals.unique()):
            out.append((k, val, (vals == val).to_numpy()))
    return out
LV_D = level_masks(D, ["ri_rule", "has_fts_data", "window"])
LV_P = level_masks(P, ["has_fts_data", "window"])
GB_P = P["sample_status"].isin(LABELLED).to_numpy()
BAD_P = (P["sample_status"] == "BAD").to_numpy()

SCORE_DIST, EXTRAP, SWAP, EXTRAP_ANY, CUTS = {}, {}, {}, {}, {}
for v in VARIANTS:
    s, M = scen(v), MODELS[scen(v)]
    nd = len(EDGES[v]) + 1
    # 3.2 declined across the approved DEV score deciles
    rows = []
    for pop, LV, dec in (("declined", LV_D, DEC_D[v]), ("approved", LV_P, DEC_P[v])):
        for kt, kv, mk in LV:
            if mk.sum() == 0:
                continue
            cnt = pd.Series(dec[mk]).value_counts().reindex(range(1, nd + 1), fill_value=0)
            for b, c in cnt.items():
                rows.append({"population": pop, "level_type": kt, "level": kv, "dev_decile": b, "n": int(c), "share": c / mk.sum(),
                             "decile_pd_upper": float(EDGES[v][b - 1]) if b <= len(EDGES[v]) else 1.0})
    SCORE_DIST[v] = pd.DataFrame(rows)

    # 3.3 extrapolation per feature
    flags, rows = np.zeros(len(D), bool), []
    for f in VFEATS[v]:
        optb, (bp, bd) = M["L1"][f], BINS[s][f]
        out_range, lo, hi = np.zeros(len(D), bool), np.nan, np.nan
        if str(getattr(optb, "dtype", "numerical")) == "numerical":
            xp = pd.to_numeric(P[f], errors="coerce").to_numpy(dtype=float)
            xd = pd.to_numeric(D[f], errors="coerce").to_numpy(dtype=float)
            if np.isfinite(xp).any():
                lo, hi = np.nanquantile(xp, EXTRAP_Q)
                with np.errstate(invalid="ignore"):
                    out_range = (xd < lo) | (xd > hi)
        sp = bp.value_counts(normalize=True)
        sparse = set(sp[sp < EXTRAP_SPARSE_BIN].index)
        in_sparse = bd.isin(sparse).to_numpy() | ~bd.isin(set(sp.index)).to_numpy()
        any_f = out_range | in_sparse
        flags |= any_f
        rows.append({"feature": f, "level_type": "ALL", "level": "ALL", "approved_p01": lo, "approved_p99": hi,
                     "n_sparse_bins": len(sparse), "share_declined_out_of_range": out_range.mean(),
                     "share_declined_in_sparse_bin": in_sparse.mean(), "share_declined_any": any_f.mean()})
    for kt, kv, mk in LV_D:
        if mk.sum():
            rows.append({"feature": "AT_LEAST_ONE_FEATURE", "level_type": kt, "level": kv, "n_declined": int(mk.sum()),
                         "share_declined_any": flags[mk].mean()})
    EXTRAP[v], EXTRAP_ANY[v] = pd.DataFrame(rows), flags

    # 3.4 swap set at proposed_cutoff, else at each approved DEV score decile of that model
    CUTS[v] = [int(PROPOSED_CUTOFF)] if PROPOSED_CUTOFF is not None else sorted(set(common.pd_to_score(EDGES[v], cfg).tolist()))
    sd_, sp_ = SC_D[v]["score"].to_numpy(), SC_P[v]["score"].to_numpy()
    pdd = SC_D[v]["pd"].to_numpy()
    LVm_D = LV_D + level_masks(D, ["app_month"])[1:]
    LVm_P = LV_P + level_masks(P, ["app_month"])[1:]
    rows = []
    for c in CUTS[v]:
        sin, sout = sd_ >= c, sp_ < c
        rate = 1 - sout.mean()
        for kt, kv, mk in LVm_D:
            if mk.sum():
                rows.append({"cutoff_score": c, "approved_rate_at_cutoff": rate, "side": "swap_in (declined, score >= cut-off)",
                             "level_type": kt, "level": kv, "n_population": int(mk.sum()), "n": int((sin & mk).sum()),
                             "share": (sin & mk).sum() / mk.sum(), "mean_pd_expected": pdd[sin & mk].mean() if (sin & mk).any() else np.nan})
        for kt, kv, mk in LVm_P:
            if mk.sum():
                g = sout & mk & GB_P
                rows.append({"cutoff_score": c, "approved_rate_at_cutoff": rate, "side": "swap_out (approved, score < cut-off)",
                             "level_type": kt, "level": kv, "n_population": int(mk.sum()), "n": int((sout & mk).sum()),
                             "share": (sout & mk).sum() / mk.sum(), "n_good_bad": int(g.sum()), "bads": int((g & BAD_P).sum()),
                             "bad_rate_observed": (g & BAD_P).sum() / g.sum() if g.any() else np.nan})
    SWAP[v] = pd.DataFrame(rows)
    print(f"{v}: extrapolation (>=1 feature) {flags.mean():.1%} | cut-offs {CUTS[v]}")

In [0]:
PAIRS = [("FTS_A", "NOFTS_A"), ("FTS_C", "NOFTS_C"), ("FTS_A", "FTS_C"), ("NOFTS_A", "NOFTS_C")]
MAIN_PAIRS = PAIRS[:2]                      # FTS vs no-FTS; the other two compare A vs C within a scenario

# 4.1 score agreement
def agreement(dx, dy, px, py):
    diff = np.abs(dx - dy)
    return {"n": len(dx), "spearman_pd": float(spearmanr(px, py).correlation) if len(dx) > 2 else np.nan,
            "diagonal_share": float((diff == 0).mean()), "diagonal_adjacent_share": float((diff <= 1).mean())}
AG, CTAB = [], []
for x, y in PAIRS:
    for pop, DEC, SC, df in (("approved", DEC_P, SC_P, P), ("declined", DEC_D, SC_D, D)):
        for kt, kv, mk in level_masks(df, ["has_fts_data"]):
            if mk.sum():
                AG.append({"pair": f"{x} vs {y}", "population": pop, "level_type": kt, "level": kv,
                           **agreement(DEC[x][mk], DEC[y][mk], SC[x]["pd"].to_numpy()[mk], SC[y]["pd"].to_numpy()[mk])})
        ct = pd.crosstab(pd.Series(DEC[x], name=f"decile_{x}"), pd.Series(DEC[y], name=f"decile_{y}")).reset_index()
        CTAB.append((f"Decile cross-table {x} (rows) vs {y} (columns), {pop}", ct))
AG = pd.DataFrame(AG)
_ap = AG[AG["population"] == "approved"].set_index(["pair", "level_type", "level"])["diagonal_adjacent_share"]
AG["approved_diagonal_adjacent_share"] = [_ap.get((p, t, l), np.nan) for p, t, l in zip(AG["pair"], AG["level_type"], AG["level"])]
AG["flag_declined_agreement_clearly_lower"] = (AG["population"] == "declined") & \
    (AG["approved_diagonal_adjacent_share"] - AG["diagonal_adjacent_share"] > AGREEMENT_FLAG_GAP)

# Cut-offs at the same approval rate on approved applicants (approve if PD <= cut-off PD)
if PROPOSED_CUTOFF is None:
    RATES = [round(r, 2) for r in np.arange(0.10, 0.91, 0.10)]
else:
    RATES = [round(float((SC_P["FTS_A"]["score"] >= PROPOSED_CUTOFF).mean()), 4)]
CUT_PD = {(v, r): float(np.quantile(SC_P[v]["pd"].to_numpy(), r)) for v in VARIANTS for r in RATES}
APP_P = {(v, r): SC_P[v]["pd"].to_numpy() <= CUT_PD[(v, r)] for v in VARIANTS for r in RATES}
APP_D = {(v, r): SC_D[v]["pd"].to_numpy() <= CUT_PD[(v, r)] for v in VARIANTS for r in RATES}

def four_groups(ax, ay, x, y):
    return {f"approved by {x} only": ax & ~ay, f"approved by {y} only": ~ax & ay, "approved by both": ax & ay, "approved by neither": ~ax & ~ay}

# 4.2 declined applicants by approve/decline group
LV_D2 = level_masks(D, ["window", "ri_rule", "has_fts_data"])
XD = []
for x, y in PAIRS:
    for r in RATES:
        G = four_groups(APP_D[(x, r)], APP_D[(y, r)], x, y)
        for kt, kv, mk in LV_D2:
            if mk.sum() == 0:
                continue
            for gname, gm in G.items():
                m_ = gm & mk
                XD.append({"pair": f"{x} vs {y}", "approval_rate_on_approved": r, "cutoff_pd_x": CUT_PD[(x, r)], "cutoff_pd_y": CUT_PD[(y, r)],
                           "level_type": kt, "level": kv, "group": gname, "n": int(m_.sum()), "n_level_total": int(mk.sum()),
                           "share": m_.sum() / mk.sum(),
                           "mean_pd_x": SC_D[x]["pd"].to_numpy()[m_].mean() if m_.any() else np.nan,
                           "mean_pd_y": SC_D[y]["pd"].to_numpy()[m_].mean() if m_.any() else np.nan})
XD = pd.DataFrame(XD)

# 4.3 swap-out comparison on approved GOOD/BAD with paired, stratified bootstrap (config.selection.bootstrap)
yb = BAD_P.astype(int)
LV_GB = [("labelled", "ALL", GB_P)] + [("window", w, GB_P & wP[w]) for w in WINDOWS if (GB_P & wP[w]).sum() > 0]
rng = np.random.default_rng(BOOT["seed"])
BIDX = {}
for kt, kv, mk in LV_GB:
    ii = np.where(mk)[0]
    if BOOT.get("stratified", True):
        ib, ig = ii[yb[ii] == 1], ii[yb[ii] == 0]
        BIDX[kv] = np.concatenate([rng.choice(ib, (BOOT["n"], len(ib))), rng.choice(ig, (BOOT["n"], len(ig)))], axis=1).astype(np.int32)
    else:
        BIDX[kv] = rng.choice(ii, (BOOT["n"], len(ii))).astype(np.int32)
XA_ = []
for x, y in PAIRS:
    for r in RATES:
        ax, ay = APP_P[(x, r)], APP_P[(y, r)]
        dec_x_only, dec_y_only = ~ax & ay, ax & ~ay
        dvec = yb * (dec_x_only.astype(int) - dec_y_only.astype(int))     # +1 = bad avoided by x only
        for kt, kv, mk in LV_GB:
            row = {"pair": f"{x} vs {y}", "approval_rate_on_approved": r, "level_type": kt, "level": kv, "n_good_bad": int(mk.sum()),
                   "bads": int((mk & BAD_P).sum())}
            for gname, gm in {f"declined by {x} only": dec_x_only, f"declined by {y} only": dec_y_only,
                              "declined by both": ~ax & ~ay, "approved by both": ax & ay}.items():
                m_ = gm & mk
                key = gname.replace(f"declined by {x} only", "declined_by_x_only").replace(f"declined by {y} only", "declined_by_y_only").replace(" ", "_")
                row[f"n_{key}"], row[f"bads_{key}"] = int(m_.sum()), int((m_ & BAD_P).sum())
                row[f"bad_rate_{key}"] = (m_ & BAD_P).sum() / m_.sum() if m_.any() else np.nan
            row["net_bads_avoided_by_x"] = int(dvec[mk].sum())
            boot = dvec[BIDX[kv]].sum(axis=1)
            row["net_ci_low"], row["net_ci_high"] = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
            row["net_per_1000_good_bad"] = 1000 * row["net_bads_avoided_by_x"] / mk.sum()
            row["x"], row["y"] = x, y
            XA_.append(row)
XA_ = pd.DataFrame(XA_)
display(AG[AG["level_type"] == "ALL"])
display(XA_[(XA_["level"] == "OOT") & XA_["pair"].isin([f"{a} vs {b}" for a, b in MAIN_PAIRS])]
        [["pair", "approval_rate_on_approved", "n_declined_by_x_only", "bad_rate_declined_by_x_only", "n_declined_by_y_only",
          "bad_rate_declined_by_y_only", "net_bads_avoided_by_x", "net_ci_low", "net_ci_high"]])

In [0]:
def watch_frame(df, src):
    out = {}
    for w, col in src.items():
        if col is None:
            continue
        if w.upper() == "AGE":
            band = pd.cut(pd.to_numeric(df[col], errors="coerce"), AGE_BANDS, right=False)
            out["AGE_BAND"] = band.astype(object).where(band.notna(), "MISSING").astype(str)
        else:
            out[w] = _norm(df[col]).astype(object).where(_norm(df[col]).notna(), "MISSING").astype(str)
    return pd.DataFrame(out, index=df.index)
WP, WD = watch_frame(P, M_WATCH), watch_frame(D, R_WATCH)
WATCH_DIMS = [c for c in WP.columns if c in WD.columns]
for c in WATCH_DIMS:
    if c != "AGE_BAND":
        top = pd.concat([WP[c], WD[c]]).value_counts().index[:TOP_N_WATCH]
        WP[c], WD[c] = WP[c].where(WP[c].isin(top), "OTHER"), WD[c].where(WD[c].isin(top), "OTHER")
SEG_AVAIL = pd.DataFrame([{"segment_column": c, "in_master": c in m_all_cols, "in_RI_table": c in ri_all_cols} for c in SEGMENTS])
SEG_DIMS = [c for c in SEGMENTS if c in m_all_cols and c in ri_all_cols]
for c in SEG_DIMS:
    WP[c], WD[c] = _norm(P[c]).fillna("MISSING").astype(str), _norm(D[c]).fillna("MISSING").astype(str)
DIMS = SEG_DIMS + WATCH_DIMS

# 4.4 through-the-door approval rate at the cut-off, overall and by watch variables
TTDA = []
for v in VARIANTS:
    for r in RATES:
        ap, ad = APP_P[(v, r)], APP_D[(v, r)]
        for dim in ["ALL"] + DIMS:
            vals = ["ALL"] if dim == "ALL" else sorted(set(WP[dim]) | set(WD[dim]))
            for val in vals:
                mp = np.ones(len(P), bool) if dim == "ALL" else (WP[dim] == val).to_numpy()
                md = np.ones(len(D), bool) if dim == "ALL" else (WD[dim] == val).to_numpy()
                n_p, n_d = int(mp.sum()), int(md.sum())
                TTDA.append({"variant": v, "approval_rate_on_approved": r, "cutoff_pd": CUT_PD[(v, r)], "dim": dim, "value": val,
                             "n_approved": n_p, "approve_rate_approved": ap[mp].mean() if n_p else np.nan,
                             "n_declined": n_d, "approve_rate_declined": ad[md].mean() if n_d else np.nan,
                             "ttd_approval_rate": (ap[mp].sum() + ad[md].sum()) / (n_p + n_d) if (n_p + n_d) else np.nan})
TTDA = pd.DataFrame(TTDA)

# 4.5 strong disagreement (|decile difference| >= 3) on declined applicants
DISAG = []
for x, y in MAIN_PAIRS:
    diff = DEC_D[x] - DEC_D[y]
    for gname, mk in {"all declined": np.ones(len(D), bool), f"{x} riskier by >= {DISAGREE_DECILES} deciles": diff >= DISAGREE_DECILES,
                      f"{y} riskier by >= {DISAGREE_DECILES} deciles": diff <= -DISAGREE_DECILES,
                      f"|decile difference| >= {DISAGREE_DECILES}": np.abs(diff) >= DISAGREE_DECILES}.items():
        n = int(mk.sum())
        DISAG.append({"pair": f"{x} vs {y}", "group": gname, "n": n, "share_of_declined": n / len(D),
                      "share_has_fts_data": (D["has_fts_data"].to_numpy()[mk] == "Y").mean() if n else np.nan,
                      "share_any_fts_model_feature": D[FTS_MODEL_FEATS].notna().any(axis=1).to_numpy()[mk].mean() if n else np.nan,
                      f"share_out_of_approved_range_{x}": EXTRAP_ANY[x][mk].mean() if n else np.nan,
                      f"share_out_of_approved_range_{y}": EXTRAP_ANY[y][mk].mean() if n else np.nan,
                      f"mean_missing_rate_{x}_features": D[VFEATS[x]].isna().mean(axis=1).to_numpy()[mk].mean() if n else np.nan,
                      f"mean_missing_rate_{y}_features": D[VFEATS[y]].isna().mean(axis=1).to_numpy()[mk].mean() if n else np.nan,
                      "mean_missing_rate_fts_features": D[FTS_MODEL_FEATS].isna().mean(axis=1).to_numpy()[mk].mean() if n else np.nan,
                      f"mean_pd_{x}": SC_D[x]["pd"].to_numpy()[mk].mean() if n else np.nan,
                      f"mean_pd_{y}": SC_D[y]["pd"].to_numpy()[mk].mean() if n else np.nan,
                      **{f"share_rule_{r}": (D["ri_rule"].to_numpy()[mk] == r).mean() if n else np.nan for r in RULES}})
DISAG = pd.DataFrame(DISAG)

# 4.6 Steps 3 and 4 by segment columns and every WATCH variable (approval-rate cut-offs)
SEGROWS = []
for dim in DIMS:
    for val in sorted(set(WP[dim]) | set(WD[dim])):
        mp, md = (WP[dim] == val).to_numpy(), (WD[dim] == val).to_numpy()
        for r in RATES:
            for v in VARIANTS:
                so = ~APP_P[(v, r)] & mp & GB_P
                SEGROWS.append({"dim": dim, "value": val, "approval_rate_on_approved": r, "item": f"{v}: swap-in / swap-out",
                                "n_declined": int(md.sum()), "swap_in": int((APP_D[(v, r)] & md).sum()),
                                "n_approved": int(mp.sum()), "swap_out": int((~APP_P[(v, r)] & mp).sum()),
                                "swap_out_good_bad": int(so.sum()), "swap_out_bad_rate": (so & BAD_P).sum() / so.sum() if so.any() else np.nan})
            for x, y in MAIN_PAIRS:
                G = four_groups(APP_D[(x, r)], APP_D[(y, r)], x, y)
                SEGROWS.append({"dim": dim, "value": val, "approval_rate_on_approved": r, "item": f"{x} vs {y}: declined groups",
                                "n_declined": int(md.sum()), **{g: int((gm & md).sum()) for g, gm in G.items()}})
SEGROWS = pd.DataFrame(SEGROWS)
print("watch dims:", WATCH_DIMS, "| segment dims available:", SEG_DIMS)
display(DISAG)

In [0]:
import mlflow

SUMMARY = []
def srow(section, item, value, detail=""):
    SUMMARY.append({"section": section, "item": item, "value": value, "detail": detail})
for r in RULES:
    srow("universe", f"rule {r}: universe / after filters / in scope",
         f"{int((uni['ri_rule'] == r).sum())} / {int((uni_in['ri_rule'] == r).sum())} / {int((D['ri_rule'] == r).sum())}")
srow("universe", "total in scope declined", len(D))
srow("universe", "approved in scope (master, not EXCLUDED)", len(P))
srow("universe", "through-the-door approval rate (approved / (approved + RI declined))", round(len(P) / (len(P) + len(D)), 4))
ra = REAPPLY.iloc[0]
srow("reapplied", "declined first, approved later (emirates_id = EMID_REFERENCENUMBER)", int(ra["declined_later_approved"]),
     f"GOOD+BAD {int(ra['later_good_bad'])}, bads {int(ra['later_bads'])}, bad rate {ra['later_bad_rate']:.4f} "
     f"vs all approved {REAPPLY_BENCH['bad_rate'].iloc[0]:.4f}")
for v in VARIANTS:
    srow("extrapolation", f"{v}: share of declined with >= 1 feature out of approved range / sparse bin", round(float(EXTRAP_ANY[v].mean()), 4))
for s in MODELS:
    srow("policy_truncated", f"{s}: policy-truncated features (overall)", len(TRUNC_FEATS.get(s, [])), ", ".join(TRUNC_FEATS.get(s, [])))
for v in VARIANTS:
    for r in RATES:
        srow("swap_in", f"{v}: declined approved at approval rate {r:.0%} on approved", int(APP_D[(v, r)].sum()))
for x, y in MAIN_PAIRS:
    t = XD[(XD["pair"] == f"{x} vs {y}") & (XD["level_type"] == "ALL")]
    for r in RATES:
        g = t[t["approval_rate_on_approved"] == r].set_index("group")["n"]
        srow("cross_swap_declined", f"{x} vs {y} at {r:.0%}: {x}-only / {y}-only approvals among declined",
             f"{int(g.get(f'approved by {x} only', 0))} / {int(g.get(f'approved by {y} only', 0))}")
for _, rw in SPOT.iterrows():
    srow("spot_check", f"{rw['variant']}: E13 reproduction (n, score diffs, max PD diff)", f"{rw['n']}, {rw['n_score_diff']}, {rw['max_abs_pd_diff']:.2e}")
SUMMARY = pd.DataFrame(SUMMARY)

META = [("module", "s09_ri_reject_fts_compare (spec Ch. 12 features-only, 6.4 PSI, E13 scoring, 8.3 paired bootstrap)"),
        ("run_time", RUN_TIME), ("git_commit", GIT_COMMIT), ("config_sha256", CONFIG_HASH), ("feature_dict_sha256", FDICT_SHA)]
for s, M in MODELS.items():
    META += [(f"{s}: registry", M["REG_PATH"]), (f"{s}: version / locked_at", f"{M['VERSION']} / {M['LOCKED_AT']}"),
             (f"{s}: registry sha256", M["REG_SHA"]), (f"{s}: selected model (status, source)", f"{M['SELECTED']} ({M['STATUS']}, {M['SELECTION']['source']}; E13 rule selection {M['SELECTION']['rule_selected_model']})"),
             (f"{s}: models scored here", ", ".join(VARIANT_MODELS)), (f"{s}: Platt file sha256", M["PLATT_SHA"]),
             (f"{s}: locked files hash-verified", M["N_FILES"]), (f"{s}: DEV GOOD+BAD for decile cut-points", N_DEV[f"{s}_A"])]
META += [("input: declined", f"{RI_TBL} (Delta v{PROTECTED_TABLES.get(RI_TBL)})"), ("input: approved", f"{MASTER_TBL} (Delta v{PROTECTED_TABLES.get(MASTER_TBL)}), sample_status != EXCLUDED"),
         ("input: overlap check", OVERLAP_TBL), ("key of declined applications", "AppKey (application_id is not unique)"),
         ("universe: decision", f"{U['decision_column']} in {DEC_VALUES} (trim + UPPER), applies to every rule"),
         ("universe: switch_date (inclusive)", U["switch_date"]), ("universe: rule A", f"{U['period_column']} < switch_date and UPPER({U['reason_column']}) contains '{REASON}'"),
         ("universe: rule B", f"{U['period_column']} >= switch_date and {U['risk_segment_column']} non-empty and in {SEG_VALUES}"),
         ("universe: rule B_FALLBACK", f"{U['period_column']} >= switch_date and {U['risk_segment_column']} empty and rule A reason"),
         ("population filters", f"mode={RI.get('population_filters_mode')}; columns not in the RI table are reported NOT_AVAILABLE (see intake_waterfall)"),
         ("windows", f"from {U['period_column']} with config.windows; outside = OUT_OF_SCOPE (reported, not used)"),
         ("derived features of declines", "as at application date (AGE = FLOOR(DATEDIFF(dectec_application_date, date_of_birth)/365.25), added to the RI table by the owner)"),
         ("has_fts_data", f"Y if non-empty {MFLAG} (approved) / {RFLAG} (declined); segment only"),
         ("cut-offs", "Step 3: approved DEV score deciles of each model (proposed_cutoff null). Step 4: same approval rate on approved applicants (10%..90%)"),
         ("bootstrap", f"{BOOT} stratified on GOOD/BAD, paired (same resamples for both models), cut-offs fixed on all approved in scope"),
         ("caveat", "DEV labels are in-sample for both models; use OOT rows for the honest swap-out comparison. FWD/STRESS labels are mostly immature"),
         ("report-only constants", f"truncation <{TRUNC_APPROVED_MAX}/>{TRUNC_DECLINED_MIN}; extrapolation {EXTRAP_Q} and bins <{EXTRAP_SPARSE_BIN}; disagreement >= {DISAGREE_DECILES}; agreement gap {AGREEMENT_FLAG_GAP}")]
META = pd.DataFrame(META, columns=["key", "value"])

SHEETS = {"meta": META, "summary": SUMMARY,
          "universe_counts": [("Totals", UNIVERSE_TOTAL), ("Monthly counts by rule", monthly)],
          "switch_diagnostics": [("From switch_date: other segments containing the rule A reason", SWITCH_DIAG),
                                 ("From switch_date: risk segment x decision", SEG_X_DEC),
                                 (f"Monthly from {SWITCH_DIAG_FROM}", monthly[monthly["app_month"] >= SWITCH_DIAG_FROM])],
          "table_overlap": TABLE_OVERLAP, "intake_waterfall": WATERFALL,
          "reasons": [("Decline reasons (report only; hard-policy flags are not excluded)", REASONS),
                      ("Declined first, approved later", REAPPLY), ("Benchmark", REAPPLY_BENCH)],
          "timing_gap": TIMING, "fts_coverage": FTS_COV}
for s in MODELS:
    for m in VARIANT_MODELS:
        v = f"{s}_{m}"
        SHEETS[f"psi_by_window_{v}"] = PSI_WIN[s][PSI_WIN[s]["feature"].isin(VFEATS[v])]
        SHEETS[f"psi_by_quarter_{v}"] = PSI_Q[s][PSI_Q[s]["feature"].isin(VFEATS[v])]
SHEETS["missing_rates"] = MISSING
SHEETS["policy_truncated"] = TRUNC.sort_values(["policy_truncated", "scenario", "feature"], ascending=[False, True, True])
SHEETS["through_the_door"] = TTD
for v in VARIANTS:
    SHEETS[f"score_distribution_{v}"] = SCORE_DIST[v]
    SHEETS[f"extrapolation_{v}"] = EXTRAP[v]
    SHEETS[f"swap_set_{v}"] = SWAP[v]
SHEETS.update({"model_agreement": [("Agreement", AG)] + CTAB, "cross_swap_declined": XD, "cross_swap_approved": XA_,
               "ttd_approval": TTDA, "disagreement_profile": DISAG,
               "segments": [("Segment columns (config.columns.segments)", SEG_AVAIL), ("By segment / watch variable", SEGROWS)]})
save_excel(SHEETS, XLSX_PATH)

n_na_seg = int((ri["_seg"].eq("NA").fillna(False).to_numpy(dtype=bool) & dec_ok & post).sum())
n_sig = int((ri["_seg"].eq("SIGNIFICANT RISK").fillna(False).to_numpy(dtype=bool) & dec_ok & post).sum())
missing_filters = WATERFALL.loc[WATERFALL["resolution"].astype(str).str.startswith("NOT_AVAILABLE"), "column_in_config"].tolist()
aliased = WATERFALL.loc[WATERFALL["resolution"].astype(str).str.startswith("alias"), ["column_in_config", "column_used"]].values.tolist()
OQ = f"""# Open questions: s09_ri_reject_fts_compare ({RUN_TIME})

1. **Shared application_id**: {int((vc > 1).sum())} application_id values appear on {int(vc[vc > 1].sum())} rows. The rows are not identical and do not differ only in FTS columns (AppKey, application/decision dates, source code and decision fields differ), so the owner's dedup rules do not apply. They are kept as separate applications keyed by AppKey. Confirm.
2. **Population filters**: {len(missing_filters)} application-time filter columns are not in the RI table and could not be applied: {missing_filters}. Aliased: {aliased}. Should these flags be added to the RI table?
3. **Risk segment value 'NA'**: {n_na_seg} declines from switch_date have the literal value 'NA'; treated as non-empty and not ELEVATED RISK (excluded from rule B and from B_FALLBACK). Also {n_sig} SIGNIFICANT RISK declines are outside rule B (config lists ELEVATED RISK only). Confirm.
4. **AGE of declines** is at dectec_application_date (owner-added column); approved AGE matches the same formula at APPROVEDATE. Confirm that application-date age is acceptable.
5. **Agreement flag**: "clearly lower" is implemented as a gap > {AGREEMENT_FLAG_GAP} in the diagonal + adjacent share (report only). Confirm the threshold.
6. **Bootstrap**: cut-offs are fixed on all approved applicants in scope; GOOD/BAD rows are resampled (stratified, paired). Confirm, or require the cut-off to be re-derived in every resample.
7. **No-FTS Model A** is PROVISIONAL in its registry (R4 pending). Results for NOFTS_A depend on that status.
8. **In-sample DEV labels**: swap-out bad rates on DEV are in-sample for both models; OOT is the honest comparison.
9. **Watch variables**: SEX is taken from 'gender' in the RI table; segment columns {SEGMENTS} do not exist in either table.
10. **has_fts_data** from the reference number understates FTS presence on approved applicants (EMID fallback): see fts_coverage.
11. **calibration/platt_dev_oof.json** is not in either registry's hash list; it was hashed at the start of this run and checked unchanged at the end.
"""
write_text(OQ, OQ_PATH)

mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="s09_ri_reject_fts_compare") as _run:
    mlflow.log_params({"config_sha256": CONFIG_HASH, "git_commit": GIT_COMMIT, "switch_date": U["switch_date"],
                       "models": ",".join(VARIANTS), **{f"{s}_registry_version": M["VERSION"] for s, M in MODELS.items()}})
    mlflow.log_metrics({"n_declined_in_scope": len(D), "n_approved_in_scope": len(P),
                        "reapplied_later_approved": int(ra["declined_later_approved"]),
                        **{f"extrapolation_share_{v}": float(EXTRAP_ANY[v].mean()) for v in VARIANTS}})
    mlflow.log_artifact(XLSX_PATH)
    mlflow.log_artifact(OQ_PATH)
    MLFLOW_RUN_ID = _run.info.run_id
print(f"Saved {XLSX_PATH} ({len(SHEETS)} sheets) and {OQ_PATH} | MLflow {MLFLOW_RUN_ID}")
display(SUMMARY)

In [0]:
CHECKS = {}
# 1. Registries, locked files, Platt files unchanged; nothing written outside the output folder
for s, M in MODELS.items():
    CHECKS[f"{s}: registry file unchanged"] = sha256_file(M["REG_PATH"]) == M["REG_SHA"]
    CHECKS[f"{s}: all locked model/binning file hashes match"] = all(
        sha256_file(v["path"]) == v["sha256"] for v in read_json(M["REG_PATH"])["files"].values())
    CHECKS[f"{s}: Platt calibration file unchanged"] = sha256_file(M["PLATT_PATH"]) == M["PLATT_SHA"]
SNAP1 = tree_snapshot([M["ROOT"] for M in MODELS.values()])
CHECKS["no file created/changed under the model folders outside the output folder"] = SNAP1 == SNAP0
CHECKS["protected Delta tables unchanged (master, RI tables, feature log)"] = {t: table_version(t) for t in PROTECTED_TABLES} == PROTECTED_TABLES
CHECKS["every file written is inside the output folder"] = all(
    os.path.abspath(p).startswith(os.path.abspath(OUT) + os.sep) for p in WRITTEN) and os.path.exists(XLSX_PATH)

# 2. Every RI row satisfies its ri_rule and the decision condition (independent re-implementation)
u = uni
d2 = pd.to_datetime(u[U["period_column"]])
dec2 = u[U["decision_column"]].fillna("").astype(str).str.strip().str.upper().isin(DEC_VALUES)
rs2 = u[U["reason_column"]].fillna("").astype(str).str.upper().str.contains(REASON, regex=False)
sg2 = u[U["risk_segment_column"]].fillna("").astype(str).str.strip().str.upper()
ok = np.select([u["ri_rule"] == "A", u["ri_rule"] == "B", u["ri_rule"] == "B_FALLBACK"],
               [(d2 < SWITCH) & rs2, (d2 >= SWITCH) & (sg2 != "") & sg2.isin(SEG_VALUES), (d2 >= SWITCH) & (sg2 == "") & rs2], False)
CHECKS["every RI row satisfies its ri_rule and the decision condition"] = bool(ok.all() and dec2.all())
r_all = ri
d3 = pd.to_datetime(r_all[U["period_column"]])
dec3 = r_all[U["decision_column"]].fillna("").astype(str).str.strip().str.upper().isin(DEC_VALUES)
rs3 = r_all[U["reason_column"]].fillna("").astype(str).str.upper().str.contains(REASON, regex=False)
sg3 = r_all[U["risk_segment_column"]].fillna("").astype(str).str.strip().str.upper()
any_rule = dec3 & (((d3 < SWITCH) & rs3) | ((d3 >= SWITCH) & (sg3 != "") & sg3.isin(SEG_VALUES)) | ((d3 >= SWITCH) & (sg3 == "") & rs3))
CHECKS["no row outside the universe satisfies a rule"] = int(any_rule.sum()) == len(uni)

# 3. Waterfall reconciles for each rule
for r in RULES:
    start, end = WF[0][f"remaining_{r}"], WF[-1][f"remaining_{r}"]
    removed = sum(row[f"removed_{r}"] for row in WF[1:])
    CHECKS[f"waterfall reconciles for rule {r}"] = (start - removed == end == int((uni_in["ri_rule"] == r).sum())
                                                   and start == int((uni["ri_rule"] == r).sum()))

# 4. Approved scores reproduce the stored E13 scores on a 1,000-row spot check
for _, rw in SPOT.iterrows():
    CHECKS[f"{rw['variant']}: E13 spot check (1,000 rows, exact score, PD within 1e-9)"] = (
        rw["n"] == 1000 and rw["n_score_diff"] == 0 and rw["max_abs_pd_diff"] <= 1e-9)

# 5. The four approve/decline groups reconcile to the totals for each model pair and cut-off
g = XD.groupby(["pair", "approval_rate_on_approved", "level_type", "level"]).agg(n=("n", "sum"), tot=("n_level_total", "first"), k=("group", "size"))
CHECKS["declined: four groups reconcile to totals (every pair, cut-off, level)"] = bool(((g["n"] == g["tot"]) & (g["k"] == 4)).all())
grp = [c for c in XA_.columns if c.startswith("n_") and c != "n_good_bad"]
CHECKS["approved GOOD/BAD: four groups reconcile to totals"] = bool((XA_[grp].sum(axis=1) == XA_["n_good_bad"]).all())
CHECKS["declined: approved-group totals equal in-scope declined count (ALL level)"] = bool(
    (XD[XD["level_type"] == "ALL"].groupby(["pair", "approval_rate_on_approved"])["n"].sum() == len(D)).all())

RES = pd.DataFrame([{"check": k, "pass": bool(v)} for k, v in CHECKS.items()])
display(RES)
ALL_PASS = bool(RES["pass"].all())
write_json({"module": "s09_ri_reject_fts_compare", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
            "registries": {s: {"path": M["REG_PATH"], "version": M["VERSION"], "sha256": M["REG_SHA"]} for s, M in MODELS.items()},
            "selected_model": {s: M["SELECTION"] for s, M in MODELS.items()},
            "rows": {"ri_table": len(ri), "universe": len(uni), "after_filters": len(uni_in), "declined_in_scope": len(D), "approved_in_scope": len(P)},
            "universe_by_rule": {r: int((D["ri_rule"] == r).sum()) for r in RULES},
            "reapplied_later_approved": REAPPLY.iloc[0].to_dict(),
            "extrapolation_share": {v: float(EXTRAP_ANY[v].mean()) for v in VARIANTS},
            "outputs": WRITTEN + [SUMMARY_PATH], "mlflow_run_id": MLFLOW_RUN_ID,
            "checks": CHECKS, "all_pass": ALL_PASS, "runtime_min": round((time.time() - _t0) / 60, 2)}, SUMMARY_PATH)
failed = RES.loc[~RES["pass"], "check"].tolist()
assert ALL_PASS, f"Acceptance FAILED: {failed}"
print(f"All {len(RES)} acceptance checks PASS | run summary {SUMMARY_PATH}")